In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../day-3/amz_uk_price_prediction_dataset.csv')

In [3]:
df.head()

,uid,asin,title,stars,reviews,price,isBestSeller,boughtInLastMonth,category
0,1,B09B96TG33,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
1,2,B01HTH3C8S,"Anker Soundcore mini, Super-Portable Bluetooth...",4.7,98099,23.99,True,0,Hi-Fi Speakers
2,3,B09B8YWXDF,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
3,4,B09B8T5VGV,"Echo Dot with clock (5th generation, 2022 rele...",4.7,7205,31.99,False,0,Hi-Fi Speakers
4,5,B09WX6QD65,Introducing Echo Pop | Full sound compact Wi-F...,4.6,1881,17.99,False,0,Hi-Fi Speakers


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2443651 entries, 0 to 2443650
Data columns (total 9 columns):
 #   Column             Dtype  
---  ------             -----  
 0   uid                int64  
 1   asin               object 
 2   title              object 
 3   stars              float64
 4   reviews            int64  
 5   price              float64
 6   isBestSeller       bool   
 7   boughtInLastMonth  int64  
 8   category           object 
dtypes: bool(1), float64(2), int64(3), object(3)
memory usage: 151.5+ MB


In [5]:
df.shape

(2443651, 9)

In [ ]:
# Create crosstab between category and isBestSeller
crosstab = pd.crosstab(df['category'], df['isBestSeller'])

# Calculate the proportion of best-sellers for each category
crosstab['best_seller_proportion'] = crosstab[True] / (crosstab[True] + crosstab[False])

# Sort categories by proportion of best-sellers in descending order
sorted_crosstab = crosstab.sort_values('best_seller_proportion', ascending=False)

# Display the top categories where being a best-seller is more prevalent
sorted_crosstab[['best_seller_proportion']].head(10)

In [ ]:
from scipy.stats import chi2_contingency

# Create contingency table
contingency_table = pd.crosstab(df['category'], df['isBestSeller'])

# Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {p:.4e}")
print(f"Degrees of freedom: {dof}")

# Compute Cramér's V
n = contingency_table.sum().sum()
min_dim = min(contingency_table.shape) - 1
cramers_v = (chi2 / (n * min_dim)) ** 0.5
print(f"Cramér's V: {cramers_v:.4f}")

In [ ]:
# Count best-seller status per category
category_bestseller_counts = df.groupby(['category', 'isBestSeller']).size().unstack(fill_value=0)

# Plot stacked bar chart
category_bestseller_counts.sort_values(by=True, ascending=False).head(15).plot(
    kind='bar', 
    stacked=True, 
    figsize=(12,6),
    color={False: 'skyblue', True: 'orange'}
)
plt.title('Best-Seller Status by Product Category (Top 15 Categories)')
plt.xlabel('Category')
plt.ylabel('Number of Products')
plt.legend(title='Best Seller')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate Q1, Q3, and IQR for the 'price' column
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for non-outlier prices
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
df_no_outliers = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]

# Display the shape of the new dataframe
df_no_outliers.shape

In [ ]:
# Top 20 categories by count
top_20_categories = df['category'].value_counts().head(20).index

plt.figure(figsize=(16, 8))
sns.violinplot(
    data=df[df['category'].isin(top_20_categories)],
    x='category',
    y='price',
    inner='quartile'
)
plt.xticks(rotation=75)
plt.title('Price Distribution by Product Category (Top 20 Categories)')
plt.xlabel('Category')
plt.ylabel('Price')
plt.tight_layout()
plt.show()

# Find category with highest median price (no filtering)
median_prices = df.groupby('category')['price'].median()
highest_median_category = median_prices.idxmax()
highest_median_price = median_prices.max()
print(f"Category with highest median price: {highest_median_category} (Median price: £{highest_median_price:.2f})")

In [ ]:
# Top 10 categories by product count
top_10_categories = df['category'].value_counts().head(10).index

# Calculate average price for top 10 categories
avg_price_top10 = df[df['category'].isin(top_10_categories)].groupby('category')['price'].mean().sort_values(ascending=False)

# Bar chart for average price of top 10 categories
plt.figure(figsize=(12, 6))
sns.barplot(x=avg_price_top10.index, y=avg_price_top10.values, palette='viridis')
plt.title('Average Price of Products for Top 10 Categories')
plt.xlabel('Category')
plt.ylabel('Average Price')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Find category with highest average price (across all categories)
avg_price_all = df.groupby('category')['price'].mean()
highest_avg_category = avg_price_all.idxmax()
highest_avg_price = avg_price_all.max()
print(f"Category with highest average price: {highest_avg_category} (Average price: £{highest_avg_price:.2f})")

In [ ]:
# Top 10 categories by product count
top_10_categories = df['category'].value_counts().head(10).index

plt.figure(figsize=(14, 7))
sns.boxplot(
    data=df[df['category'].isin(top_10_categories)],
    x='category',
    y='stars'
)
plt.title('Distribution of Product Ratings by Category (Top 10 Categories)')
plt.xlabel('Category')
plt.ylabel('Product Rating (Stars)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Find category with highest median rating (across all categories)
median_ratings = df.groupby('category')['stars'].median()
highest_median_category = median_ratings.idxmax()
highest_median_rating = median_ratings.max()
print(f"Category with highest median rating: {highest_median_category} (Median rating: {highest_median_rating:.2f})")

In [ ]:
# Calculate the correlation coefficient between 'price' and 'stars'
correlation = df['price'].corr(df['stars'])
print(f"Correlation coefficient between price and stars: {correlation:.4f}")

# Interpret the result
if abs(correlation) < 0.1:
    print("There is no significant correlation between product price and its rating.")
elif abs(correlation) < 0.3:
    print("There is a weak correlation between product price and its rating.")
elif abs(correlation) < 0.5:
    print("There is a moderate correlation between product price and its rating.")
else:
    print("There is a strong correlation between product price and its rating.")

In [ ]:
# Scatter plot: Product rating vs. price
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='stars', y='price', alpha=0.3)
plt.title('Scatter Plot: Product Rating vs. Price')
plt.xlabel('Product Rating (Stars)')
plt.ylabel('Price')
plt.tight_layout()
plt.show()

# Correlation heatmap for all numerical variables
plt.figure(figsize=(8, 6))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numerical Variables')
plt.tight_layout()
plt.show()

# QQ plot for price distribution
import scipy.stats as stats

plt.figure(figsize=(6, 6))
stats.probplot(df['price'].dropna(), dist="norm", plot=plt)
plt.title('QQ Plot of Product Prices')
plt.tight_layout()
plt.show()

from IPython.display import display, Markdown

report = """
# Business-Centric Report: Amazon UK Price Prediction Dataset

## 1. Dataset Overview
- **Total Products:** {:,}
- **Total Categories:** {:,}
- **Columns:** {}
- The dataset contains detailed information on Amazon UK products, including price, ratings, reviews, best-seller status, and category.

## 2. Best-Seller Insights
- Certain categories have a higher proportion of best-sellers. Top categories by best-seller prevalence can guide inventory and marketing focus.
- **Statistical Association:** There is a statistically significant association between product category and best-seller status (Chi-square test, see analysis above).
- **Cramér's V:** {:.4f} (indicates {} association between category and best-seller status).

## 3. Price Analysis
- **Outlier Handling:** Price outliers were filtered using the IQR method for robust analysis.
- **Category Price Distribution:** Some categories have notably higher median and average prices.
    - **Highest Median Price Category:** {} (£{:.2f})
    - **Highest Average Price Category:** {} (£{:.2f})
- **Top 10 Categories:** Visualizations show significant price variation across popular categories.

## 4. Ratings Analysis
- **Category Ratings:** Product ratings (stars) vary by category.
    - **Highest Median Rating Category:** {} ({:.2f} stars)
- **Price vs. Rating Correlation:** Correlation coefficient between price and rating is {:.4f}, indicating {} correlation.

## 5. Recommendations
- **Focus on High Best-Seller Categories:** Prioritize inventory and marketing for categories with high best-seller proportions.
- **Premium Pricing Opportunities:** Explore pricing strategies in categories with higher median/average prices.
- **Customer Satisfaction:** Maintain high product ratings, especially in top-performing categories.
- **Further Analysis:** Consider deeper dives into review text, seasonal trends, and competitor pricing for actionable insights.

---

*This report summarizes key findings from the exploratory data analysis. For detailed visualizations and statistical outputs, refer to the notebook cells above.*
""".format(
    len(df),
    df['category'].nunique(),
    list(df.columns),
    cramers_v,
    "strong" if cramers_v > 0.3 else "moderate" if cramers_v > 0.1 else "weak",
    highest_median_category, highest_median_price,
    highest_avg_category, highest_avg_price,
    highest_median_category, highest_median_rating,
    correlation,
    "no significant" if abs(correlation) < 0.1 else "weak" if abs(correlation) < 0.3 else "moderate" if abs(correlation) < 0.5 else "strong"
)

display(Markdown(report))